# PHY 657 — Problem Set 4: Linear Classification
**Spring 2026**

Mo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(42)
plt.rcParams['figure.figsize'] = (7, 5)

---
## Problem 1: Generating Data

Generate two 2D Gaussian clusters with identity covariance, separated by $\mu$ along the x-axis.

In [ ]:
def generate_data(mu, n=500):
    """Generate two 2D Gaussian clusters."""
    x0 = np.random.multivariate_normal([0, 0], np.eye(2), n)
    x1 = np.random.multivariate_normal([mu, 0], np.eye(2), n)
    X = np.vstack([x0, x1])
    y = np.array([0]*n + [1]*n)
    return X, y

# generate for mu = 3
X, y = generate_data(mu=3)

plt.scatter(X[y==0, 0], X[y==0, 1], alpha=0.5, label='Class 0', s=15)
plt.scatter(X[y==1, 0], X[y==1, 1], alpha=0.5, label='Class 1', s=15)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Two Gaussian Clusters ($\\mu = 3$)')
plt.legend()
plt.axis('equal')
plt.tight_layout()
plt.show()

---
## Problem 2: Linear Classifier

Fit logistic regression and show the decision boundary + probability regions.

In [ ]:
def plot_classifier(X, y, mu_val):
    """Fit logistic regression, plot decision boundary and probability shading."""
    clf = LogisticRegression()
    clf.fit(X, y)

    # create a mesh for probability shading
    margin = 1.5
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    # shade by predicted probability
    plt.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.6)
    plt.colorbar(label='P(class 1)')
    # decision boundary at P = 0.5
    plt.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    # data points
    plt.scatter(X[y==0, 0], X[y==0, 1], alpha=0.4, label='Class 0', s=12)
    plt.scatter(X[y==1, 0], X[y==1, 1], alpha=0.4, label='Class 1', s=12)
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(f'Logistic Regression ($\\mu = {mu_val}$)')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

    return clf

clf = plot_classifier(X, y, 3)

---
## Problem 3: Performance vs Separation

Sweep over $\mu = 3, 2, 1, 0.5$ and compute accuracy, FPR, FNR.

In [ ]:
mu_values = [3, 2, 1, 0.5]
results = {'mu': [], 'accuracy': [], 'fpr': [], 'fnr': []}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for i, mu in enumerate(mu_values):
    X_i, y_i = generate_data(mu)
    clf_i = LogisticRegression()
    clf_i.fit(X_i, y_i)
    y_pred = clf_i.predict(X_i)

    # metrics
    acc = accuracy_score(y_i, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_i, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    results['mu'].append(mu)
    results['accuracy'].append(acc)
    results['fpr'].append(fpr)
    results['fnr'].append(fnr)

    print(f'mu = {mu:.1f}  |  Accuracy = {acc:.3f}  |  FPR = {fpr:.3f}  |  FNR = {fnr:.3f}')

    # mini plot with decision boundary
    ax = axes[i]
    margin = 1.5
    x_min, x_max = X_i[:, 0].min() - margin, X_i[:, 0].max() + margin
    y_min, y_max = X_i[:, 1].min() - margin, X_i[:, 1].max() + margin
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = clf_i.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.5)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1.5)
    ax.scatter(X_i[y_i==0, 0], X_i[y_i==0, 1], alpha=0.3, s=8)
    ax.scatter(X_i[y_i==1, 0], X_i[y_i==1, 1], alpha=0.3, s=8)
    ax.set_title(f'$\\mu = {mu}$,  Acc = {acc:.2f}')
    ax.set_xlabel('$x_1$')
    if i == 0:
        ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

In [ ]:
# accuracy vs mu plot
plt.figure(figsize=(6, 4))
plt.plot(results['mu'], results['accuracy'], 'o-', color='tab:blue', linewidth=2, markersize=8)
plt.xlabel('$\\mu$ (class separation)')
plt.ylabel('Accuracy')
plt.title('Classifier Accuracy vs Class Separation')
plt.ylim(0.4, 1.05)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Problem 4: Interpretation

**1. Why does performance degrade as $\mu \to 0$?**

As $\mu \to 0$ the two class distributions overlap more and more. When $\mu = 0$ the two Gaussians are identical, so there is literally no information in the features to tell the classes apart. No classifier — linear or otherwise — can do better than random guessing (50%) in that limit.

**2. Does adding more data help when $\mu = 0$?**

No. When $\mu = 0$ the class-conditional distributions are the same ($p(\mathbf{x}|y=0) = p(\mathbf{x}|y=1)$). More data just gives you a better estimate of a boundary that doesn't exist. The Bayes-optimal accuracy is 50% regardless of sample size.

**3. What assumption does a linear classifier make about the data?**

A linear classifier assumes the two classes can be separated by a hyperplane (a straight line in 2D). Equivalently, it assumes the log-odds of class membership is a linear function of the features: $\log \frac{P(y=1|\mathbf{x})}{P(y=0|\mathbf{x})} = \mathbf{w}^T \mathbf{x} + b$. This is actually the correct model here since both classes are Gaussians with the same covariance, so logistic regression is well-suited to this problem.